In [2]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
import zipfile

from nerfstudio.field_components.field_heads import FieldHeadNames
from nerfstudio.fields.base_field import Field, get_normalized_directions
from nerfstudio.utils.eval_utils import eval_setup
import torch

img_names = [
    "frame_00003.JPG",
    "frame_00005.JPG"
]

device = "cuda:0"

In [ ]:
# load the model
# TODO: change the config path to your own path
config_path = Path(".../outputs/nerfacto/2024-12-12_092010/config.yml")
config, pipeline, checkpoint_path, _ = eval_setup(config_path)

# Inspect the dataset

In this section, we will inspect the dataset used for testing.
We will:
1. Load the test dataset and cameras
2. Visualize the test images (2 images) to understand what we're working with
3. Print dataset information like length and available data fields



In [ ]:
datamanager = pipeline.datamanager
dataset = datamanager.eval_dataset
cameras = dataset.cameras
print(f"len of dataset: {len(dataset)}")
datas = [dataset.get_data(i) for i in range(len(dataset))]
print(datas[0].keys())

# visualize the test images (2 images) in separate plots (since they are in different sizes)

images = [datas[i]['image'] for i in range(2)]
images = [np.array(image) for image in images]
images = [image for image in images]

for i, image in enumerate(images):
    plt.imshow(image)
    plt.show()


# Test run

Before we start to refine the latents, let's first test run the model to see how it works. Let's use the first image as an example.

The process from camera pose to rendered image:
1. Generate rays from camera pose using camera.generate_rays()
   - Each ray has origin and direction based on camera parameters

2. Process rays in chunks to avoid memory issues
   - Split ray bundle into smaller chunks (4096 rays each)
   - Pass each chunk through the NeRF model
   
3. Model processes each ray to produce RGB output (The NeRF pipeline)
   - Samples points along each ray
   - Evaluates density and color at sampled points
   - Accumulates samples to get final pixel color
   - *Please notice that the NeRFacto model is slightly different from the original NeRF paper. It uses proposal sampler instead of the original coarse-to-fine sampling strategy. See https://docs.nerf.studio/nerfology/methods/nerfacto.html for more details.*

4. Combine chunk outputs into full image
   - Concatenate RGB values from all chunks
   - Reshape into final image dimensions (height x width x 3)


In [ ]:
torch.cuda.empty_cache()
model = pipeline.model.to("cuda")
camera = cameras[0]
rays = camera.generate_rays(camera_indices=0, keep_shape=True, obb_box=None).to("cuda")

@torch.no_grad()
def get_outputs_for_camera_ray_bundle(model, camera_ray_bundle):
    """Takes in camera parameters and computes the output of the model.
    adapted from nerfstudio/nerfstudio/models/base_model.py

    Args:
        camera_ray_bundle: ray bundle to calculate outputs over
    """
    num_rays_per_chunk = 4096
    image_height, image_width = camera_ray_bundle.origins.shape[:2]
    num_rays = len(camera_ray_bundle)
    outputs_lists = defaultdict(list)
    print(f"num_rays: {num_rays}, num_rays_per_chunk: {num_rays_per_chunk}, chunk_num: {num_rays // num_rays_per_chunk}")
    for i in tqdm(range(0, num_rays, num_rays_per_chunk)):
        start_idx = i
        end_idx = i + num_rays_per_chunk
        ray_bundle = camera_ray_bundle.get_row_major_sliced_ray_bundle(start_idx, end_idx)
        # move the chunk inputs to the model device
        ray_bundle = ray_bundle.to(model.device)
        outputs = model.forward(ray_bundle=ray_bundle)
        for output_name, output in outputs.items():  # type: ignore
            if not isinstance(output, torch.Tensor):
                continue
            # move the chunk outputs from the model device back to the device of the inputs.
            outputs_lists[output_name].append(output.to("cpu"))
    outputs = {}
    for output_name, outputs_list in outputs_lists.items():
        outputs[output_name] = torch.cat(outputs_list).view(image_height, image_width, -1)  # type: ignore
    return outputs

field_outputs = get_outputs_for_camera_ray_bundle(model, rays)
print(field_outputs.keys())

image = field_outputs['rgb'].cpu().numpy()
plt.imshow(image)
plt.show()

# Analysis

As you can see, the model is able to render the image, but the rendered image do not match the ground truth in time.

Let's try to refine the latents to see if we can get a better result.


In [ ]:
image_height, image_width = rays.origins.shape[:2]

########################################################
# TODO : get top half of the rays
########################################################

top_rays = ...

field_outputs = get_outputs_for_camera_ray_bundle(model, top_rays)
image = field_outputs['rgb'].cpu().numpy()
plt.imshow(image)
plt.show()

gt = images[0][:image_height // 2, ...]
plt.imshow(gt)
plt.show()

As can be seen in above, the rendered image looks as if is captured in daytime, while the ground truth is captured in nighttime.

This is because the model use a zero embedding for the appearance at test time.

Let's go and see how we can refine the latents.

We will first walk you through the pipeline of the model.

## Ray sampling

As the first step, we need to sample rays from the camera, so that we do not run out of memory.

We will sample 4096 rays from the camera ray bundle.


In [8]:
# sample rays
num_sampled_rays = 4096
sampled_indices = torch.tensor(np.random.choice(len(rays), num_sampled_rays, replace=False)).cuda()
ray_bundle = rays.flatten()[sampled_indices]

field_outputs = model.forward(ray_bundle=ray_bundle)
image = field_outputs['rgb'].detach().cpu().numpy()

## Model forward

The model forward is the core of the NeRF pipeline. Please refer to https://github.com/nerfstudio-project/nerfstudio/blob/555d5540086cc6e85717be6b07cc37d5d07af893/nerfstudio/models/nerfacto.py#L298 for the original implementation.

The rough steps are:
1. Pass the rays through the proposal sampler to get the ray samples.
2. Pass the ray samples through the NeRF model to get the rgb value for each ray point.
3. Compute the weights for the ray samples. 
4. Compute the rgb for the ray samples. (Alpha blending)


In [9]:
ray_samples, weights_list, ray_samples_list = model.proposal_sampler(ray_bundle, density_fns=model.density_fns)
field_outputs = model.field.forward(ray_samples, compute_normals=model.config.predict_normals)
print(field_outputs.keys())
weights = ray_samples.get_weights(field_outputs[FieldHeadNames.DENSITY])
rgb = model.renderer_rgb(rgb=field_outputs[FieldHeadNames.RGB], weights=weights)

### Appearance embedding

However, we didn't see how the appearance embedding is used in the model forward.

Let's go deeper into the field query process and see how the appearance embedding is used.

The following code is from https://github.com/nerfstudio-project/nerfstudio/blob/555d5540086cc6e85717be6b07cc37d5d07af893/nerfstudio/fields/nerfacto_field.py#L234

In [10]:
field = model.field
density, density_embedding = field.get_density(ray_samples)

assert density_embedding is not None
if ray_samples.camera_indices is None:
    raise AttributeError("Camera indices are not provided.")
camera_indices = ray_samples.camera_indices.squeeze()
directions = get_normalized_directions(ray_samples.frustums.directions)
directions_flat = directions.view(-1, 3)
d = field.direction_encoding(directions_flat)

outputs_shape = ray_samples.frustums.directions.shape[:-1]

# appearance
embedded_appearance = None
if field.embedding_appearance is not None:
    if field.training:
        embedded_appearance = field.embedding_appearance(camera_indices)
    else:
        if field.use_average_appearance_embedding:
            embedded_appearance = torch.ones(
                (*directions.shape[:-1], field.appearance_embedding_dim), device=directions.device
            ) * field.embedding_appearance.mean(dim=0)
        else:
            embedded_appearance = torch.zeros(
                (*directions.shape[:-1], field.appearance_embedding_dim), device=directions.device
            )

h = torch.cat(
    [
        d,
        density_embedding.view(-1, field.geo_feat_dim),
    ]
    + (
        [embedded_appearance.view(-1, field.appearance_embedding_dim)] if embedded_appearance is not None else []
    ),
    dim=-1,
)
rgb = field.mlp_head(h).view(*outputs_shape, -1).to(directions)


As can be seen, the appearance embedding is concatenated to the other features before is feed into the MLP head.

In [ ]:
field.embedding_appearance

# Your term
Now that I have walked you through the model forward, it is your turn to implement the refinement process.

Let's initialize a new appearance embedding by random and the optimizer.

In [ ]:
new_embedding = torch.nn.Embedding(1, 32).cuda()
optimizer = torch.optim.Adam(new_embedding.parameters(), lr=1e-3)

We then implement a function that allows you to take customized appearance embedding and get the field outputs.

In [12]:
def get_field_outputs(field, ray_samples, appearance_embedding):
    density, density_embedding = field.get_density(ray_samples)

    ###########################################################
    # TODO: implement the field query process
    ###########################################################

    rgb = field.mlp_head(h).view(*outputs_shape, -1).to(directions)
    return density, rgb


Given this function, we can then implement the refinement process.

The idea is to keep the trained model fixed and only refine the appearance embedding via reconstruction loss.

The rough steps are:
1. Sample rays from the camera.
2. Pass the rays and the appearance embedding through the model to get the field outputs.
3. Compute the reconstruction loss.
4. Backpropagate the loss and update the appearance embedding.


In [ ]:
num_iters = 1000
num_rays = 4096
for i in tqdm(range(num_iters)):
    optimizer.zero_grad()
    # sample rays
    sampled_indices = torch.tensor(np.random.choice(len(rays), num_rays, replace=False)).cuda()
    ray_bundle = rays.flatten()[sampled_indices]
    ray_bundle = model.collider(ray_bundle)

    # sample ray points
    # TODO: sample the ray points

    # get field outputs
    # TODO: render the RGB for each sampled pixel

    ray_gt = torch.tensor(images[0]).cuda()[:image_height // 2, ...].reshape(-1, 3)[sampled_indices]
    # TODO: compute the loss
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"iter {i}, loss: {loss.item()}")


After refining the appearance embedding, we can then evaluate the result.

In [ ]:
# evaluate using the new embedding
rays = camera.generate_rays(camera_indices=0, keep_shape=True, obb_box=None).to("cuda")

with torch.no_grad():
    image_height, image_width = rays.origins.shape[:2]
    num_rays = len(rays)
    rgb_list = []
    for i in tqdm(range(0, num_rays, num_sampled_rays)):
        start_idx = i
        end_idx = i + num_sampled_rays
        ray_bundle = rays.get_row_major_sliced_ray_bundle(start_idx, end_idx)
        # move the chunk inputs to the model device
        ray_bundle = ray_bundle.to(model.device)
        ray_bundle = model.collider(ray_bundle)
        
        # TODO: you can copy the code from the previous block to here
        
        rgb_list.append(rgb.to("cpu"))
    rgb = torch.cat(rgb_list).view(image_height, image_width, -1)  # type: ignore

plt.imshow(rgb.cpu().numpy())
plt.show()

Now save your result (and dont forget to do the same thing for the second image) and submit it to the leaderboard!

Also you are encouraged to tune the parameters (learning rate, number of iterations, etc.) to get a better result.

In [ ]:
# save image
from PIL import Image
output_dir = Path("refined")
output_dir.mkdir(parents=True, exist_ok=True)
rgb = rgb.cpu().numpy() * 255
rgb = rgb.astype(np.uint8)
Image.fromarray(rgb).save(output_dir / img_names[0])

In [ ]:
zip_path = Path("refined.zip")

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for image_file in img_names:
        file_path = output_dir / image_file
        if file_path.exists():
            zipf.write(file_path, image_file)
        else:
            print(f"Warning: {image_file} not found in {output_dir}")

print(f"Created zip file: {zip_path}")